# Factor Analysis — Momentum & Value

This notebook computes the 12-1 momentum factor and the composite value factor across the default universe, then runs Information Coefficient (IC) analysis to evaluate their raw predictive power at multiple forward-return horizons.

See `src/alpha_engine/research/ic_analysis.py` for the underlying IC/ICIR implementation.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import matplotlib.pyplot as plt

from alpha_engine.config import DEFAULT_UNIVERSE
from alpha_engine.data.price_loader import PriceLoader
from alpha_engine.data.fundamental_loader import FundamentalLoader
from alpha_engine.factors.momentum import compute_12_1_momentum
from alpha_engine.factors.value import compute_composite_value
from alpha_engine.research.ic_analysis import multi_horizon_ic
from alpha_engine.research.factor_decay import compute_ic_decay, half_life

print(f'Universe: {DEFAULT_UNIVERSE}')

In [ ]:
loader = PriceLoader()
prices = loader.get_close_prices(DEFAULT_UNIVERSE, start='2019-01-01', end='2024-01-01')
prices.tail()

## 1. Momentum Factor (12-1)

In [ ]:
momentum_factor = compute_12_1_momentum(prices)
momentum_factor.tail()

In [ ]:
momentum_ic_table = multi_horizon_ic(momentum_factor, prices, horizons=(1, 5, 10, 21))
momentum_ic_table

In [ ]:
decay_curve = compute_ic_decay(momentum_factor, prices, max_horizon=42)
print(f'Estimated IC half-life: {half_life(decay_curve)} trading days')

fig, ax = plt.subplots(figsize=(8, 4))
decay_curve.plot(ax=ax, marker='o', markersize=3)
ax.set_title('Momentum Factor — IC Decay Curve')
ax.set_xlabel('Forward horizon (trading days)')
ax.set_ylabel('Mean Rank IC')
ax.grid(alpha=0.3)
plt.show()

## 2. Value Factor (Composite: Earnings Yield + Book-to-Price)

In [ ]:
fundamental_loader = FundamentalLoader()
fundamentals = fundamental_loader.get_fundamentals_frame(DEFAULT_UNIVERSE)
fundamentals

In [ ]:
value_scores = compute_composite_value(fundamentals)

# Value fundamentals here are point-in-time snapshots (not a full history),
# so we broadcast the same cross-section across all dates to demonstrate the
# IC methodology. In production, point-in-time fundamentals history would be
# stored per rebalance date instead.
value_factor_panel = pd.DataFrame(
    [value_scores.reindex(prices.columns).values] * len(prices),
    index=prices.index,
    columns=prices.columns,
)

value_ic_table = multi_horizon_ic(value_factor_panel, prices, horizons=(1, 5, 10, 21))
value_ic_table

## 3. Comparing Momentum vs Value ICIR

In [ ]:
comparison = pd.DataFrame({
    'momentum_icir': momentum_ic_table['icir'],
    'value_icir': value_ic_table['icir'],
})

fig, ax = plt.subplots(figsize=(8, 4))
comparison.plot(kind='bar', ax=ax)
ax.set_title('ICIR by Forward-Return Horizon: Momentum vs Value')
ax.set_xlabel('Horizon (trading days)')
ax.set_ylabel('ICIR')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

comparison

## Conclusion

This mirrors the standard factor research workflow used throughout the `alpha_engine` package: compute a cross-sectional factor, evaluate its Rank IC across multiple horizons, check ICIR for consistency, and study the decay curve to decide the appropriate rebalance frequency. See `scripts/run_factor_research.py` for a CLI version of this workflow across the full factor zoo, and `README.md` for the Factor Zoo reference table.